In [1]:

%%shell 
git clone https://github.com/FallenChromium/lm-polygraph.git
cd lm-polygraph
git checkout experiments/tfb

Cloning into 'lm-polygraph'...
remote: Enumerating objects: 10115, done.
remote: Counting objects: 100% (92/92), done.
remote: Compressing objects: 100% (61/61), done.
remote: Total 10115 (delta 54), reused 37 (delta 30), pack-reused 10023 (from 2)
Receiving objects: 100% (10115/10115), 15.76 MiB | 11.57 MiB/s, done.
Resolving deltas: 100% (6503/6503), done.
Branch 'experiments/tfb' set up to track remote branch 'experiments/tfb' from 'origin'.
Switched to a new branch 'experiments/tfb'


In [1]:
%cd lm-polygraph
!pip install numpy==1.26
!pwd
!ls

/content/lm-polygraph
/content/lm-polygraph
CONTRIBUTING.md   docs	      notebooks       requirements.txt	test
dataset_builders  examples    pyproject.toml  scripts		uv.lock
Dockerfile	  LICENSE.md  README.md       src


In [2]:
!git pull
!uv add -r requirements.txt
import sys
import site
# Add the uv venv's site-packages to the path
venv_site_packages = "/content/lm-polygraph/.venv/lib/python3.12/site-packages"
if venv_site_packages not in sys.path:
    sys.path.insert(0, venv_site_packages)
# Also tell site about it
site.addsitedir(venv_site_packages)

Already up to date.
Resolved 176 packages in 0.91ms
Audited 168 packages in 2ms


In [3]:
print(sys.path)
!python --version
import numpy as np
print(f"NumPy version: {np.__version__}")

['/content/lm-polygraph/.venv/lib/python3.12/site-packages', '/content', '/env/python', '/usr/lib/python312.zip', '/usr/lib/python3.12', '/usr/lib/python3.12/lib-dynload', '', '/usr/local/lib/python3.12/dist-packages', '/usr/lib/python3/dist-packages', '/usr/local/lib/python3.12/dist-packages/IPython/extensions', '/root/.ipython', '/content/lm-polygraph/src']
Python 3.12.12
NumPy version: 1.26.0


# TFB (Training-Free Bayesianization) Example

This notebook demonstrates how to use TFB for uncertainty estimation with LoRA-finetuned models.

TFB converts pre-trained LoRA weights into Bayesian posteriors via SVD-based variance inference,
enabling stochastic sampling without additional training.

**Reference**: Shi et al. "Training-Free Bayesianization for Low-Rank Adapters of Large Language Models" (arXiv:2412.05723)

## Requirements
- TFB **requires trained LoRA weights** (non-zero B matrix). Freshly initialized LoRA won't produce diverse samples.
- For best results, use a LoRA adapter that was fine-tuned on your task.

In [4]:
import torch
import numpy as np
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel, LoraConfig, get_peft_model
from datasets import load_dataset

from lm_polygraph import WhiteboxModel
from lm_polygraph.utils.tfb import (
    apply_tfb, 
    enable_tfb_sampling, 
    disable_tfb_sampling,
    fit_tfb_beta,
    _extract_lora_layers,
)
from lm_polygraph.estimators.tfb import TFBPredictiveEntropy, TFBSampleVariance
from lm_polygraph.stat_calculators.tfb_sample import TFBSamplingCalculator
from lm_polygraph.utils.generation_parameters import GenerationParameters

## Load a Model with Trained LoRA

For meaningful TFB results, you need a model with **trained** (non-zero) LoRA-B weights.
Options:
1. Use a pre-trained LoRA from HuggingFace Hub
2. Train your own LoRA adapter
3. For demo: Initialize with small random values (shows mechanism, not realistic uncertainty)

In [5]:
# Small model for demo - works on Kaggle/Colab free tier

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen2.5-0.5B-Instruct")
model = PeftModel.from_pretrained(model, "ShahzebKhoso/qwen2.5-instruct-0.5B-pubmedqa-lora")
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-0.5B-Instruct")

# CRITICAL: Move model to device BEFORE applying TFB
model = model.to(device)

# Set padding token if not already set
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"Model loaded on {device}")

Using device: cuda


/content/lm-polygraph/.venv/lib/python3.12/site-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


adapter_config.json:   0%|          | 0.00/881 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

`low_cpu_mem_usage` was None, now default to True since model is quantized.


...

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Model loaded on cuda


In [6]:

model.print_trainable_parameters()

trainable params: 0 || all params: 495,114,112 || trainable%: 0.0000


## Load PubMedQA Dataset

Load the PubMedQA dataset that the model was fine-tuned on. We'll use the `pqa_labeled` subset which contains questions requiring yes/no/maybe answers based on medical abstracts.

In [7]:
# Load PubMedQA dataset
dataset = load_dataset("qiaojin/PubMedQA", "pqa_labeled", split="train")

# Take a subset for calibration and testing
calibration_samples = dataset.select(range(0, 250))  # 250 samples for calibration
test_samples = dataset.select(range(260, 270))  # 10 samples for testing

print(f"Loaded {len(calibration_samples)} calibration samples")
print(f"Loaded {len(test_samples)} test samples")

# Show example structure
print("\nExample question:")
example = test_samples[0]
print(f"Question: {example['question']}")
# print(f"Context: {example['context'][:200]}...")
print(f"Expected answer: {example['final_decision']}")

README.md: 0.00B [00:00, ?B/s]

pqa_labeled/train-00000-of-00001.parquet:   0%|          | 0.00/1.08M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1000 [00:00<?, ? examples/s]

Loaded 250 calibration samples
Loaded 10 test samples

Example question:
Question: Does Residency Selection Criteria Predict Performance in Orthopaedic Surgery Residency?
Expected answer: yes


## Apply TFB and Verify Setup

TFB performs SVD on each LoRA-B matrix and computes variance parameters for noise injection.

In [8]:
# Apply TFB - this modifies LoRA layers in-place
BETA = 0.013438  # Noise scale. Higher = more diverse samples
lora_layers = apply_tfb(model, beta=BETA)
print(f"Applied TFB to {len(lora_layers)} LoRA layers")

# Verify TFB attributes were added
layer = lora_layers[0]
print(f"\nLayer has TFB attributes:")
print(f"  - tfb_sampling_enabled: {layer.tfb_sampling_enabled}")
print(f"  - tfb_beta: {layer.tfb_beta}")
print(f"  - lora_A_rho shape: {layer.lora_A_rho['default'].shape}")

Applied TFB to 48 LoRA layers

Layer has TFB attributes:
  - tfb_sampling_enabled: False
  - tfb_beta: 0.013438
  - lora_A_rho shape: torch.Size([16, 896])


## Verify Stochastic Behavior

When TFB sampling is enabled, the same input should produce different outputs.

In [9]:
# Format a PubMedQA question for the model
def format_pubmed_question(sample):
    """Format PubMedQA sample using Qwen's chat template."""
    context = sample['context']
    question = sample['question']
    
    # Use Qwen's chat template format
    messages = [
        {
            "role": "system",
            "content": "You are a medical expert. Answer the question with 'yes', 'no', or 'maybe' based on the provided context."
        },
        {
            "role": "user",
            "content": f"Context: {context}\n\nQuestion: {question}\n\nAnswer with 'yes', 'no', or 'maybe':"
        }
    ]
    
    # Apply chat template
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    return prompt

test_input = format_pubmed_question(test_samples[0])
inputs = tokenizer(test_input, return_tensors="pt", truncation=True, max_length=512).to(device)

model.eval()

# Deterministic (sampling disabled)
disable_tfb_sampling(model)
with torch.no_grad():
    out1 = model(**inputs).logits
    out2 = model(**inputs).logits
print(f"Deterministic: outputs identical? {torch.allclose(out1, out2)}")

# Stochastic (sampling enabled)
enable_tfb_sampling(model)
outputs = []
with torch.no_grad():
    for i in range(5):
        out = model(**inputs).logits
        outputs.append(out.clone())

# Check variance across samples
stacked = torch.stack(outputs)
variance = stacked.var(dim=0).mean().item()
print(f"Stochastic: mean variance across samples: {variance:.6f}")
print(f"Stochastic: all same? {all(torch.allclose(outputs[0], o) for o in outputs[1:])}")

if variance < 1e-6:
    print("\n⚠️ WARNING: Low variance! Check that LoRA-B has non-zero weights.")
else:
    print(f"\n✓ TFB sampling working correctly (variance = {variance:.6f})")

Deterministic: outputs identical? True
Stochastic: mean variance across samples: 1.950195
Stochastic: all same? False

✓ TFB sampling working correctly (variance = 1.950195)


## Generate Samples with TFB

Use `TFBSamplingCalculator` to generate multiple samples from the posterior.

In [10]:
# Wrap model for LM-Polygraph
gen_params = GenerationParameters(do_sample=False, max_new_tokens=50)
lm_model = WhiteboxModel(model, tokenizer, generation_parameters=gen_params)

# Generate TFB samples for a PubMedQA question
calculator = TFBSamplingCalculator(n_samples=20, beta=0.01)
enable_tfb_sampling(model)

sample = test_samples[0]
question = format_pubmed_question(sample)
stats = calculator({}, [question], lm_model, max_new_tokens=50)

print(f"Question: {sample['question']}")
print(f"Expected: {sample['final_decision']}")
print(f"\nTFB Samples:")
for i, (text, lp) in enumerate(zip(stats['tfb_sample_texts'][0], stats['tfb_sample_log_probs'][0])):
    print(f"  {i+1}. (log_prob={lp:.2f}) {text.strip()}")

Question: Does Residency Selection Criteria Predict Performance in Orthopaedic Surgery Residency?
Expected: yes

TFB Samples:
  1. (log_prob=-21.89) Yes, I can provide you with the information you need to complete the task.
  2. (log_prob=-3.52) Yes,
  3. (log_prob=-47.39) The orthopaedic residency selection criteria for the American Society for Orthopaedic Surgery (AASE) Examination (ASEQ) are not predictive of performance in the AASEQ.
  4. (log_prob=-23.80) Yes, I am a medical expert and I can provide a response based on the context.
  5. (log_prob=-5.86) No.
  6. (log_prob=-39.97) 'Residency Selection Criteria, Selection, Selection, Residency Selection, Selection, Selection, Selection, Selection, Selection, Selection, Selection, Selection, Selection, Selection, Selection, Selection, Selection, Selection, Selection, Selection, Selection, Selection,
  7. (log_prob=-13.87) No, "yes, "yes, "no, or "maybe"":
  8. (log_prob=-5.55) yes, Orthopaedic Surgery
  9. (log_prob=-1.89) Yes.
  10.

In [11]:
# Compute uncertainty estimates
entropy_est = TFBPredictiveEntropy()
variance_est = TFBSampleVariance()

entropy = entropy_est(stats)[0]
variance = variance_est(stats)[0]

print(f"Uncertainty metrics:")
print(f"  Predictive Entropy: {entropy:.4f}")
print(f"  Sample Variance: {variance:.4f}")

Uncertainty metrics:
  Predictive Entropy: 27.6751
  Sample Variance: 550.4975


## Compare Uncertainty Across Different PubMedQA Questions

Analyze uncertainty estimates on various medical questions. Questions with ambiguous or complex medical scenarios should show higher uncertainty.

In [19]:
# Analyze uncertainty across multiple PubMedQA samples
print(f"{'Question':<80} {'Expected':<10} {'Entropy':<12} {'Variance':<12}")
print("-" * 115)

for sample in test_samples.select(range(10)):
    question = format_pubmed_question(sample)
    stats = calculator({}, [question], lm_model, max_new_tokens=50)
    entropy = entropy_est(stats)[0]
    variance = variance_est(stats)[0]
    
    # Truncate question for display
    q_display = sample['question'][:75] + "..." if len(sample['question']) > 75 else sample['question']
    expected = sample['final_decision']
    
    print(f"{q_display:<80} {expected:<10} {entropy:<12.4f} {variance:<12.4f}")

Question                                                                         Expected   Entropy      Variance    
-------------------------------------------------------------------------------------------------------------------
Does Residency Selection Criteria Predict Performance in Orthopaedic Surger...   yes        13.8369      262.9314    
Optimism and survival: does an optimistic outlook predict better survival a...   yes        24.2071      369.4510    
Is it better to be big?                                                          no         11.7273      173.2321    
Is arch form influenced by sagittal molar relationship or Bolton tooth-size...   no         15.5015      310.3234    
Cold knife conization vs. LEEP. Are they the same procedure?                     no         12.7597      213.0442    
Are pectins involved in cold acclimation and de-acclimation of winter oil-s...   yes        15.4567      192.2248    
Updating emotional content in working memory: a depression

## Optional: Calibrate Beta

The `fit_tfb_beta` function finds optimal beta that maximizes sample diversity
while keeping prediction degradation below a threshold.

In [12]:
# Prepare calibration data from PubMedQA
calibration_texts = [
    format_pubmed_question(sample) 
    for sample in calibration_samples.select(range(0,250))  # Use 5 samples for quick calibration
]
calibration_inputs = [
    tokenizer(text, return_tensors='pt', truncation=True, max_length=512).to(device)
    for text in calibration_texts
]

print(f"Using {len(calibration_inputs)} PubMedQA samples for calibration")

# Find optimal beta (quick calibration for demo)
optimal_beta = fit_tfb_beta(
    model,
    calibration_inputs,
    target_metric_ratio=0.01,  # 1% degradation target
    max_iters=8,
    n_samples=50,
    initial_beta=0.1,
    verbose=True,
)
print(f"\nOptimal beta: {optimal_beta:.6f}")

Using 250 PubMedQA samples for calibration
Baseline metric: 1.043945
Iter 0: beta=0.050500, metric=8.015625, change_ratio=0.026719
Iter 1: beta=0.025750, metric=4.179688, change_ratio=0.012016


## Summary

**TFB workflow demonstrated on PubMedQA:**
1. Load model with **trained** LoRA adapter (this model is fine-tuned on PubMedQA)
2. `apply_tfb(model, beta)` - transforms LoRA for Bayesian sampling via SVD
3. `enable_tfb_sampling(model)` - activates stochastic forward passes
4. Use `TFBSamplingCalculator` to generate posterior samples
5. Use `TFBPredictiveEntropy` / `TFBSampleVariance` for uncertainty scores
6. Calibrate beta with `fit_tfb_beta` using domain-specific data

**Key parameters:**
- `beta`: Noise scale (0.01-0.3 typical). Higher = more diverse samples
- `n_samples`: Number of posterior samples (5-20 typical)

**Expected behavior:**
- Medical questions with clear yes/no answers should show **lower uncertainty**
- Ambiguous or complex medical scenarios should show **higher uncertainty**
- The uncertainty estimates can help identify when the model is less confident in its predictions